> **Revisão documental — 20/10/2026:** Saídas históricas preservadas. C45 conserva IDs como strings, embora a consulta atual use inteiros. Logs de instalação, benchmark e busca anteriores não validam a escala 0,5–5 e os parâmetros da fonte atual. Execução integral desta revisão no Colab pendente.

# Sistema de recomendação

Baseado no exemplo de Kuldeep Singh disponível em:
(https://github.com/singhsidhukuldeep/Recommendation-System)

**Como construir um sistema de recomendação usando scikit-surprise (biblioteca surprise)**

Duas das formas mais populares de abordar os sistemas de recomendação são [ltragem
colaborativa] e [recomendações baseadas em conteúdo].

Neste tutorial/exemplo, vamos nos concentrar na abordagem de **filtragem colaborativa**, ou seja:
o usuário recomenda itens que pessoas com gostos e preferências semelhantes gostavam no
passado. Em outras palavras, esse método prevê classicações desconhecidas usando as
semelhanças entre os usuários.

A carga usa a versão 1 de `snehal1409/movielens`, conferida em 2026-10-19,
com IDs brutos inteiros e notas na escala de 0,5 a 5. O modelo final recebe
todos os parâmetros selecionados por RMSE. A consulta ilustrativa exige que
o usuário e o item existam individualmente no treino; o par não precisa
ter uma avaliação observada.

As saídas salvas são históricas e foram preservadas. As correções de escala,
IDs e transferência dos parâmetros exigem nova execução antes de atribuir
notas ou métricas atualizadas ao exemplo.


## Instalando as bibliotecas necessárias

Surprise é um scikit Python para construir e analisar sistemas de recomendação.
http://surpriselib.com/

Surprise fornece vários algoritmos de previsão prontos para uso, como algoritmos de linha de
base, métodos de vizinhança (k-nn), baseados em fatoração de matriz (SVD, PMF, SVD ++, NMF)
e muitos outros. Além disso, várias medidas de similaridade (cosseno, MSD, pearson ...) estão
embutidas. Facilite a implementação de novas idéias de algoritmos. Fornece ferramentas para
avaliar, analisar e comparar o desempenho dos algoritmos. Os procedimentos de validação
cruzada podem ser executados facilmente usando poderosos iteradores de CV (inspirados nas
excelentes ferramentas do scikit-learn), bem como uma pesquisa exaustiva sobre um conjunto
de parâmetros. O nome SurPRISE signica Simple Python RecommendatIon System Engine.

Observe que a surpresa não oferece suporte a classicações implícitas ou informações
baseadas em conteúdo.


In [ ]:
# Surprise 1.1.5 tem suporte a NumPy 2; não é necessário forçar downgrade.
%pip install -q "scikit-surprise==1.1.5"

from importlib.metadata import version
import sys

if "surprise" in sys.modules and sys.modules["surprise"].__version__ != version("scikit-surprise"):
    raise RuntimeError("Reinicie a sessão e execute as células novamente para carregar a versão instalada de Surprise.")

print({pacote: version(pacote) for pacote in ("numpy", "scipy", "scikit-surprise")})


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 8.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 74.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompat

## Preparação e compatibilidade do Scikit-Surprise

Execute a célula anterior antes dos imports. Ela instala `scikit-surprise==1.1.5`, que admite NumPy 2, e informa as versões instaladas de NumPy, SciPy e Surprise. Não é necessário forçar um downgrade de NumPy nem encerrar automaticamente o processo da sessão.

Se uma versão antiga de Surprise já estiver carregada, a preparação exibirá uma mensagem solicitando a reinicialização manual da sessão. Nesse caso, reinicie e execute novamente as células iniciais. A instalação não encerra o kernel por conta própria.

Os antigos procedimentos com `numpy<2.0` e encerramento do processo foram substituídos pela preparação acima. Se aparecerem conflitos de dependências, confira as versões e o resultado das importações; não os trate genericamente como avisos inofensivos.

As saídas históricas deste notebook não foram recalculadas nesta manutenção.

## Importando bibliotecas necessárias

In [2]:
import pandas as pd
from surprise import Reader
from surprise import Dataset
from surprise.model_selection import cross_validate
#preditores da biblioteca
from surprise import NormalPredictor
from surprise import KNNBasic
from surprise import KNNWithMeans
from surprise import KNNWithZScore
from surprise import KNNBaseline
from surprise import SVD
from surprise import BaselineOnly
from surprise import SVDpp
from surprise import NMF
from surprise import SlopeOne
from surprise import CoClustering
# metricas de avaliacao
from surprise.accuracy import rmse
from surprise import accuracy
from surprise.model_selection import train_test_split
from surprise.model_selection import GridSearchCV

## Coletando os dados

O GroupLens Research coletou e disponibilizou conjuntos de dados de classicação do site
MovieLens ([MovieLens](https://movielens.org/)). Os conjuntos de dados foram
coletados em vários períodos de tempo, dependendo do tamanho do conjunto.

Estamos usando um arquivo Pequeno: 100.000 classicações e 3.600 aplicativos de tag
aplicados a 9.000 lmes por 600 usuários. Última atualização em 9/2018.

Download: [ml-latest-small.zip](https://files.grouplens.org/datasets/movielens/ml-latest-small.zip) (tamanho: 1 MB)

In [7]:
# Dependência para carregar o CSV pelo adaptador Pandas.
%pip install -q "kagglehub[pandas-datasets]"
import pandas as pd
from pandas.api.types import is_integer_dtype, is_numeric_dtype
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "ratings.csv"
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "snehal1409/movielens/versions/1",
    file_path,
)

required_columns = {"userId", "movieId", "rating", "timestamp"}
if not required_columns.issubset(df.columns):
    raise ValueError("Colunas inesperadas no MovieLens")
if df.empty or df[list(required_columns)].isna().any().any():
    raise ValueError("MovieLens sem registros ou com valores ausentes")
if not all(is_integer_dtype(df[column]) for column in ["userId", "movieId"]):
    raise ValueError("Os IDs brutos do MovieLens devem ser inteiros")
if not is_numeric_dtype(df["rating"]) or not df["rating"].between(0.5, 5).all():
    raise ValueError("As notas do MovieLens devem estar entre 0,5 e 5")

print("Primeiros 5 registros:", df.head())

Using Colab cache for faster access to the 'movielens' dataset.
First 5 records:    userId  movieId  rating   timestamp
0       1       31     2.5  1260759144
1       1     1029     3.0  1260759179
2       1     1061     3.0  1260759182
3       1     1129     2.0  1260759185
4       1     1172     4.0  1260759205


In [10]:
ratings = df.copy()

O arquivo ratings possui os campos:

* userId (usuário que fez a avaliação)
* movieId (lme avaliado)
* rating (avaliação)
* timestamp (quando foi feita a avaliação)

In [11]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [12]:
# não precisamos a marca do tempo
ratings.drop(['timestamp'], axis=1, inplace=True)
ratings.columns = ['userID', 'item', 'rating']

In [13]:
ratings.head()

,userID,item,rating
0,1,31,2.5
1,1,1029,3.0
2,1,1061,3.0
3,1,1129,2.0
4,1,1172,4.0


In [14]:
print("Quantidade de avaliações: ", ratings.shape[0])

Quantidade de avaliações:  100004


## Limpeza da base: Cold Start

Para evitar problemas de "cold start" eliminaremos da nossa base os lmes que quase nunca
foram avaliados

In [15]:
# filtramos os itens que tiveram mais do que 5 avaliações
min_ratings = 5
filter_items = ratings['item'].value_counts() > min_ratings
filter_items = filter_items[filter_items].index.tolist()

In [16]:
# filtramos os itens que tiveram mais do que 5 usuários avaliando
min_user_ratings = 5
filter_users = ratings['userID'].value_counts() > min_user_ratings
filter_users = filter_users[filter_users].index.tolist()

In [17]:
# removemos da base aqueles que não atendem a essas seleções
ratings_new = ratings[(ratings['item'].isin(filter_items)) & (ratings['userID'].isin(filter_users))]

In [18]:
print('Quantidade original de dados:\t{}'.format(ratings.shape[0]))
print('Quantidade após o filtro:\t{}'.format(ratings_new.shape[0]))

Quantidade original de dados:	100004
Quantidade após o filtro:	88087


## Surprise

Para carregar um conjunto de dados de um dataframe do pandas, usaremos o método load_from_df (), também precisaremos de um objeto Reader e o parâmetro rating_scale deve ser especicado. O dataframe deve ter três colunas, correspondendo aos ids do usuário, os ids dos itens e as classicações nesta ordem. Cada linha corresponde, portanto, a uma determinada classicação.


In [19]:
if ratings_new.empty or ratings_new["rating"].isna().any() or not ratings_new["rating"].between(0.5, 5).all():
    raise ValueError("Notas filtradas incompatíveis com a escala de 0,5 a 5")
reader = Reader(rating_scale=(0.5, 5))
data = Dataset.load_from_df(ratings_new[['userID', 'item', 'rating']], reader)

## Treinando e Otimizando os modelos

#### NormalPredictor

O algoritmo NormalPredictor prevê uma classicação aleatória com base na distribuição do
conjunto de treinamento, que é considerado normal. Este é um dos algoritmos mais básicos
que não funcionam muito.

#### BaselineOnly

O algoritmo BasiclineOnly prevê a estimativa da linha de base para determinado usuário e item.

#### Algoritmos k-NN

**KNNBasic** é um algoritmo básico de ltragem colaborativa.
**KNNWithMeans** é um algoritmo básico de ltragem colaborativa, levando em consideração as
avaliações médias de cada usuário.
**KNNWithZScore** é um algoritmo básico de ltragem colaborativa, levando em consideração a
normalização do z-score de cada usuário.
**KNNBaseline** é um algoritmo básico de ltragem colaborativa que leva em consideração uma
classicação de linha de base.

#### Algoritmos baseados em fatoração de matriz

**SVD** O algoritmo SVD é equivalente a Probabilistic Matrix Factorization
(http://papers.nips.cc/paper/3208-probabilistic-matrix-factorization.pdf)
**SVDpp** O algoritmo SVDpp é uma extensão do SVD que leva em consideração as classicações
implícitas.
**NMF** é um algoritmo de ltragem colaborativa baseado na Fatoração de Matrizes Não
Negativas. É muito semelhante ao SVD.
**Slope One** Slope One é uma implementação direta do algoritmo SlopeOne.
(https://arxiv.org/abs/cs/0702144)
**Co-clustering** Co-clustering é um algoritmo de ltragem colaborativa baseado em co-clustering
(http://citeseerx.ist.psu.edu/viewdoc/download?doi=10.1.1.113.6458&rep=rep1&type=pdf)

Usamos RMSE como nossa métrica de precisão para as previsões.


In [20]:
benchmark = []

In [24]:
# algoritmos que serão testados
algorithms = [SVD(), SVDpp(), SlopeOne(), NMF(), NormalPredictor(), KNNBaseline(), KNNBasic()]

In [26]:
# treinando e avaliando cada algoritmo
for algorithm in algorithms:
 print("Iniciando: " ,str(algorithm.__class__.__name__))
 # validação cruzada
 results = cross_validate(algorithm, data, measures=['RMSE'], cv=3, verbose=False)
 # coleta dos resultados
 tmp = pd.DataFrame.from_dict(results).mean(axis=0)
 tmp = pd.concat([tmp, pd.Series([str(algorithm).split(' ')[0].split('.')[-1]], index=['Algorithm'])])
 benchmark.append(tmp)

Iniciando:  SVD
Iniciando:  SVDpp
Iniciando:  SlopeOne
Iniciando:  NMF
Iniciando:  NormalPredictor
Iniciando:  KNNBaseline
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
Iniciando:  KNNBasic
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.


In [27]:
print ('\n\tPronto\n')


	Pronto



In [28]:
# construimos um DataFrame ordenado pelo valor do RMSE
surprise_results = pd.DataFrame(benchmark).set_index('Algorithm').sort_values('test_rmse')

In [29]:
surprise_results

,test_rmse,fit_time,test_time
Algorithm,,,
SVDpp,0.874653,31.022541,10.296318
KNNBaseline,0.877544,0.350433,3.654499
SVD,0.886435,1.567659,0.238077
SlopeOne,0.901054,1.533857,6.422420
NMF,0.923146,1.877850,0.212545
KNNBasic,0.941390,0.122358,1.901834
NormalPredictor,1.415418,0.073504,0.212229


## Otimização de Parâmetros

Escolhemos o SVD ou outro que possua um bom resultado de RMSE. Então realizamos um
GridSearchCV para otimizar os parâmetros.

In [ ]:
# param_grid = {
# "n_epochs": [5, 10, 15, 20, 30, 40, 50, 100],
# "lr_all": [0.001, 0.002, 0.005],
# "reg_all": [0.02, 0.08, 0.4, 0.6]
# }

In [30]:
# smaller grid for testing
param_grid = {
 "n_epochs": [10, 20],
 "lr_all": [0.002, 0.005],
 "reg_all": [0.02]
}

In [31]:
gs = GridSearchCV(SVD, param_grid, measures=["rmse", "mae"], refit=True, cv=5)

In [32]:
gs.fit(data)

In [33]:
training_parameters = gs.best_params["rmse"].copy()

In [34]:
print("Melhor RMSE: \t", gs.best_score["rmse"])
print("Melhor MAE: \t", gs.best_score["mae"])
print("Melhor parâmetros: \t", gs.best_params["rmse"])

Melhor RMSE: 	 0.8784806705608403
Melhor MAE: 	 0.6755939006902164
Melhor parâmetros: 	 {'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}


## Criando o modelo

Utilizando os melhores parâmetros obtidos criamos nosso modelo

In [35]:
from datetime import datetime
print(training_parameters)
reader = Reader(rating_scale=(0.5, 5))

{'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.02}


In [36]:
print("\n\n\t\t Inicio\n\n")
start = datetime.now()



		 Inicio




In [37]:
print("> Carregando dados...")
data = Dataset.load_from_df(ratings_new[['userID', 'item', 'rating']], reader)
print("> OK")

> Carregando dados...
> OK


In [38]:
print("> Criando base de treinamento...")
trainset = data.build_full_trainset()
print("> OK")

> Criando base de treinamento...
> OK


In [40]:
print("> treinando o modelo...")
algo = SVD(**training_parameters)
algo.fit(trainset)

> treinando o modelo...


In [41]:
end = datetime.now()
print (">> Finalizado em", (end-start).seconds, "segundos" )

>> Finalizado em 40 segundos


In [42]:
# predicitng
from pprint import pprint as pp

In [43]:
def itemRating(user, item):
 # Este exemplo consulta IDs brutos conhecidos; o par pode ainda não ter avaliação.
 trainset.to_inner_uid(user)
 trainset.to_inner_iid(item)
 prediction = algo.predict(user, item, verbose=True)
 rating = prediction.est
 details = prediction.details
 uid = prediction.uid
 iid = prediction.iid
 true = prediction.r_ui
 ret = {
    'user': user,
    'item': item,
    'rating': rating,
    'details': details,
    'uid': uid,
    'iid': iid,
    'true' : true
 }
 return ret

In [44]:
#prevendo qual a avaliação que um usuário daria para um item
print("Avaliação: ", itemRating(user = 610, item = 10))

user: 610        item: 10         r_ui = None   est = 3.58   {'was_impossible': False}
Avaliação:  {'user': '610', 'item': '10', 'rating': 3.5843086948130827, 'details': {'was_impossible': False}, 'uid': '610', 'iid': '10', 'true': None}
